# MuscleMap comparison: first notebook

## load the mha files for the MyosegmenTUM dataset

In [ ]:
# libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from ipywidgets import interact, fixed
from IPython.display import clear_output
import SimpleITK as sitk

## examine image example- 
Here we will examine a single segmentation and our grount truth we will make by combining the mha files

In [ ]:
# Load images for ground truth segmentation
img1 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_L_GR.mha") #left gracilis
img2 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_L_HS.mha")# left hamstrings
img3 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_L_QF.mha")# left quadracepts femoris
img4 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_L_SA.mha") #left  sartorius
img5 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_R_GR.mha") #right gracilis
img6 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_R_HS.mha") # right hamstring
img7 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_R_QF.mha")
img8 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack1_R_SA.mha") # right sartorius
# Convert to arrays
arr1 = sitk.GetArrayFromImage(img1)
arr2 = sitk.GetArrayFromImage(img2)
arr3 = sitk.GetArrayFromImage(img3)
arr4 = sitk.GetArrayFromImage(img4)
arr5 = sitk.GetArrayFromImage(img5)
arr6 = sitk.GetArrayFromImage(img6)
arr7 = sitk.GetArrayFromImage(img7)
arr8 = sitk.GetArrayFromImage(img8)
# Create empty label map
combined_gt = arr1 * 0
# Assign labels (1, 2, 3...)
combined_gt[arr1 > 0] = 1
combined_gt[arr2 > 0] = 2
combined_gt[arr3 > 0] = 3
combined_gt[arr4 > 0] = 4
combined_gt[arr5 > 0] = 5
combined_gt[arr6 > 0] = 6
combined_gt[arr7 > 0] = 7
combined_gt[arr8 > 0] = 8
# # Convert back to image?
out = sitk.GetImageFromArray(combined_gt)
out.CopyInformation(img1) # keep spacing/origin

In [ ]:
## examine a file
# note images should plot with radiological convention
def display_2images(fatfrac_image_z, segment_image_z, fatfrac_npa, segment_npa):
    # Create a figure with two subplots and the specified size.
    plt.subplots(1,2,figsize=(10,8))
    # Draw the fixed image in the first subplot.
    plt.subplot(1,2,1)
    plt.imshow(fatfrac_npa[fatfrac_image_z,:,:],cmap=plt.cm.Greys_r)
    plt.title('fat fraction image')
    plt.axis('off')
    # Draw the moving image in the second subplot.
    plt.subplot(1,2,2)
    vmins = 0
    vmaxs = 8
    plt.imshow(segment_npa[segment_image_z,:,:],cmap='Greens', vmin=vmins, vmax=vmaxs)
    plt.title('ground truth')
    plt.axis('off')
    
    plt.show()
    
fatfrac_name = "../myosegmenTUM/HV001_1/ImageData/HV001_1_FATFRACTION/HV001_1_FATFRACTION_stack1.nii"  
fatfrac_image = sitk.ReadImage(fatfrac_name)
segment_image= out
interact(
    display_2images,
    fatfrac_image_z = (0,fatfrac_image.GetSize()[2]-1),
    segment_image_z = (0,segment_image.GetSize()[2]-1),
    fatfrac_npa = fixed(sitk.GetArrayViewFromImage(fatfrac_image)),
    segment_npa = fixed(sitk.GetArrayViewFromImage(segment_image)))

In [ ]:
def display_overlay(z, fatfrac_npa, segment_npa, cmap1, cmap2):
    plt.figure(figsize=(6,6))

    # Base image (grayscale)
    plt.imshow(fatfrac_npa[z,:,:], cmap=cmap1)

    # Overlay (segmentation)
    plt.imshow(segment_npa[z,:,:],
               cmap=cmap2,
               alpha=0.4,     # transparency
               vmin=0, vmax=8)

    plt.title(f"Overlay (z={z})")
    plt.axis('off')
    plt.show()

In [ ]:
interact(
    display_overlay,
    z=(0, fatfrac_image.GetSize()[2]-1),
    fatfrac_npa=fixed(sitk.GetArrayViewFromImage(fatfrac_image)),
    segment_npa=fixed(sitk.GetArrayViewFromImage(segment_image)),
    cmap1='gray',
    cmap2='Greens',
)

In [ ]:
# Ground truth seems eroded ? maybe the new segmentation is better?

## now let's examine this with our machine generated image (here from MuscleMap)

In [ ]:

# Optional: discrete green colormap for labels
greens = plt.cm.Greens(np.linspace(0.3, 1, 9))
reds = plt.cm.Reds(np.linspace(0.3, 1, 9))
cmap_seg = ListedColormap(greens)
cmap_gt = ListedColormap(reds)

def display_images(fatfrac_image_z, segment_image_z, gt_image_z,
                   fatfrac_npa, segment_npa, gt_npa):

    plt.figure(figsize=(15, 8))

    # Fat fraction
    plt.subplot(1, 3, 1)
    plt.imshow(fatfrac_npa[fatfrac_image_z,:,:], cmap='Greys_r')
    plt.title('fat fraction image R,L')
    plt.axis('off')

    # Segmentation (prediction)
    plt.subplot(1, 3, 2)
    plt.imshow(segment_npa[segment_image_z,:,:],
               cmap=cmap_seg, vmin=0, vmax=8)
    plt.title('segmentation (prediction)')
    plt.axis('off')

    # Ground truth
    plt.subplot(1, 3, 3)
    plt.imshow(gt_npa[gt_image_z,:,:],
               cmap=cmap_gt, vmin=0, vmax=8)
    plt.title('ground truth segmentation')
    plt.axis('off')

    plt.show()

In [ ]:
segment_image = sitk.ReadImage("../MuscleMap_segs/HV001_1_FATFRACTION_stack1_dseg.nii")

gt_image= out
interact(
    display_images,
    fatfrac_image_z=(0, fatfrac_image.GetSize()[2]-1),
    segment_image_z=(0, segment_image.GetSize()[2]-1),
    gt_image_z=(0, gt_image.GetSize()[2]-1),

    fatfrac_npa=fixed(sitk.GetArrayViewFromImage(fatfrac_image)),
    segment_npa=fixed(sitk.GetArrayViewFromImage(segment_image)),
    gt_npa=fixed(sitk.GetArrayViewFromImage(gt_image))
)

## Uh oh, we don't have the same things segmented!

Muscle maps segments each muscle, wheras our ground truth is over groups...


Our ground truth has:\
left gracilis\
left hamstrings\
left quadricepts femoris\
left  sartorius\
right gracilis\
right hamstring\
right hamstrings\
right quadricepts femoris\

dictionary from https://musclemap.github.io/MuscleMap/muscle-anatomy/\

includes
QUADRICPETS:
thigh 	vastus lateralis 	left 	7101\
thigh 	vastus lateralis 	right 	7102\
thigh 	vastus intermedius 	left 	7111\
thigh 	vastus intermedius 	right 	7112\
thigh 	vastus medialis 	left 	7121\
thigh 	vastus medialis 	right 	7122\
thigh 	rectus femoris 	left 	7131\
thigh 	rectus femoris 	right 	7132\
SARTORIUS\
thigh 	sartorius 	left 	7141\
thigh 	sartorius 	right 	7142\
GRACILIS\
thigh 	gracilis 	left 	7151\
thigh 	gracilis 	right 	7152\
HAMSTRINGS =\
thigh 	semimembranosus 	left 	7161\
thigh 	semimembranosus 	right 	7162
thigh 	semitendinosus 	left 	7171\
thigh 	semitendinosus 	right 	7172\
thigh 	biceps femoris long head 	left 	7181\
thigh 	biceps femoris long head 	right 	7182\
thigh 	biceps femoris short head 	left 	7191\
thigh 	biceps femoris short head 	right 	7192\

# so first let's make only these, then merge them?

## note we predicted more muscles than we have ground truth for...let's take out the muscles we predicted that we have no ground truth for

In [ ]:
new_segment_image = segment_image
new_segment_array = sitk.GetArrayFromImage(new_segment_image)

In [ ]:
#np.unique(new_segment_array)

In [ ]:
# keep only keep labels we care about
keep_labels = [
    7101, 7102,
    7111, 7112,
    7121, 7122,
    7131, 7132,
    7141, 7142,
    7151, 7152,
    7161, 7162,
    7171, 7172,
    7181, 7182,
    7191, 7192
]

mask = np.isin(new_segment_array, keep_labels)
filtered = np.where(mask, new_segment_array, 0)

In [ ]:
filtered_image = sitk.GetImageFromArray(filtered)
filtered_image.CopyInformation(gt_image)

In [ ]:
# show new selection
interact(
    display_images,
    fatfrac_image_z=(0, fatfrac_image.GetSize()[2]-1),
    segment_image_z=(0, filtered_image.GetSize()[2]-1),
    gt_image_z=(0, gt_image.GetSize()[2]-1),

    fatfrac_npa=fixed(sitk.GetArrayViewFromImage(fatfrac_image)),
    segment_npa=fixed(sitk.GetArrayViewFromImage(filtered_image)),
    gt_npa=fixed(sitk.GetArrayViewFromImage(gt_image))
)

In [ ]:
interact(
    display_overlay,
    z=(0, fatfrac_image.GetSize()[2]-1),
    fatfrac_npa=fixed(sitk.GetArrayViewFromImage(gt_image)),
    segment_npa=fixed(sitk.GetArrayViewFromImage(filtered_image)),
    cmap1='Reds',
    cmap2='Greens',
)

In [ ]:
## OK, we have quirck in the system..let'3s therefore just examine gracilis and sartorius because quadricepts seem MORE accurate to reality...

## we may need to add volume similarity- because there will be a trend toward larger or smaller than ground truth?

In [ ]:
#gt_image =

In [ ]:
# right gracilis
gt = sitk.Cast(gt_image ==  5, sitk.sitkUInt8)
pred = sitk.Cast(filtered_image == 7152, sitk.sitkUInt8)

dice_filter = sitk.LabelOverlapMeasuresImageFilter()
dice_filter.Execute(gt, pred)

right_grac_dice_upper = dice_filter.GetDiceCoefficient()
# dice_filter.GetJaccardCoefficient()
# dice_filter.GetVolumeSimilarity()
# dice_filter.GetFalseNegativeError()
# dice_filter.GetFalsePositiveError()
print("Right gracilis upper dice:", right_grac_dice_upper)
hd_filter = sitk.HausdorffDistanceImageFilter()
hd_filter.Execute(gt, pred)

right_grac_hd_upper = hd_filter.GetHausdorffDistance()
print("Right gracilis upper Hausdorff distance:", right_grac_hd_upper)

In [ ]:
# left gracilis
gt = sitk.Cast(gt_image ==  1, sitk.sitkUInt8)
pred = sitk.Cast(filtered_image == 7151, sitk.sitkUInt8)

dice_filter = sitk.LabelOverlapMeasuresImageFilter()
dice_filter.Execute(gt, pred)

left_grac_dice_upper = dice_filter.GetDiceCoefficient()
print("Left gracilis upper dice:", left_grac_dice_upper)
hd_filter = sitk.HausdorffDistanceImageFilter()
hd_filter.Execute(gt, pred)

left_grac_hd_upper = hd_filter.GetHausdorffDistance()
print("Left gracilis upper Hausdorff distance:", left_grac_hd_upper)

# we will avoid average haussfdorf's because that POTENTIALLY rewards a very jumpy line that goes in and out...unless we use the average of absolute values., but still let's not complicate things

## let's look at the second part

In [ ]:
# Load images for ground truth segmentation
img1 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_L_GR.mha") #left gracilis
img2 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_L_HS.mha")# left hamstrings
img3 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_L_QF.mha")# left quadracepts femoris
img4 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_L_SA.mha") #left  sartorius
img5 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_R_GR.mha") #right gracilis
img6 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_R_HS.mha") # right hamstring
img7 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_R_QF.mha")
img8 = sitk.ReadImage("../myosegmenTUM/HV001_1/SegmentationMasks/HV001_1_stack2_R_SA.mha") # right sartorius
# Convert to array2
arr1 = sitk.GetArrayFromImage(img1)
arr2 = sitk.GetArrayFromImage(img2)
arr3 = sitk.GetArrayFromImage(img3)
arr4 = sitk.GetArrayFromImage(img4)
arr5 = sitk.GetArrayFromImage(img5)
arr6 = sitk.GetArrayFromImage(img6)
arr7 = sitk.GetArrayFromImage(img7)
arr8 = sitk.GetArrayFromImage(img8)
# Create empty label map
combined_gt = arr1 * 0
# Assign labels (1, 2, 3...)
combined_gt[arr1 > 0] = 1
combined_gt[arr2 > 0] = 2
combined_gt[arr3 > 0] = 3
combined_gt[arr4 > 0] = 4
combined_gt[arr5 > 0] = 5
combined_gt[arr6 > 0] = 6
combined_gt[arr7 > 0] = 7
combined_gt[arr8 > 0] = 8
# # Convert back to image?
out2 = sitk.GetImageFromArray(combined_gt)
out2.CopyInformation(img1) # keep spacing/origin

In [ ]:
np.unique(combined_gt)#out2

In [ ]:
gt_image2 = out2
fatfrac_image = sitk.ReadImage("../myosegmenTUM/HV001_1/ImageData/HV001_1_FATFRACTION/HV001_1_FATFRACTION_stack2.nii")
segment_image = sitk.ReadImage("../MuscleMap_segs/HV001_1_FATFRACTION_stack2_dseg.nii")
interact(
    display_images,
    fatfrac_image_z=(0, fatfrac_image.GetSize()[2]-1),
    segment_image_z=(0, segment_image.GetSize()[2]-1),
    gt_image_z=(0, out2.GetSize()[2]-1),

    fatfrac_npa=fixed(sitk.GetArrayViewFromImage(fatfrac_image)),
    segment_npa=fixed(sitk.GetArrayViewFromImage(segment_image)),
    gt_npa=fixed(sitk.GetArrayViewFromImage(gt_image2))
)

In [ ]:
# gt filter gt_image

In [ ]:
#new_segment_image = segment_image
#print(type(new_segment_image))
new_segment_array2 = sitk.GetArrayFromImage(segment_image)

In [ ]:
keep_labels = [
    7101, 7102,
    7111, 7112,
    7121, 7122,
    7131, 7132,
    7141, 7142,
    7151, 7152,
    7161, 7162,
    7171, 7172,
    7181, 7182,
    7191, 7192
]

mask = np.isin(new_segment_array2, keep_labels)
filtered = np.where(mask, new_segment_array2, 0)
filtered_image = sitk.GetImageFromArray(filtered)
filtered_image.CopyInformation(gt_image2)

In [ ]:
segment_image.CopyInformation(filtered_image)
#gt_image2 =filtered_image

In [ ]:
print(gt_image2.GetSize(), segment_image.GetSize())
print(gt_image2.GetSpacing(), segment_image.GetSpacing())
print(gt_image2.GetOrigin(), segment_image.GetOrigin())
print(gt_image2.GetDirection(), segment_image.GetDirection())

In [ ]:
# actually lower
results = []
gt = sitk.Cast(gt_image2 ==  5, sitk.sitkUInt8)
pred = sitk.Cast(segment_image == 7152, sitk.sitkUInt8)

gt_arr = sitk.GetArrayFromImage(gt)
pred_arr = sitk.GetArrayFromImage(pred)

print("GT voxels:", np.sum(gt_arr))
print("Pred voxels:", np.sum(pred_arr))
dice_filter = sitk.LabelOverlapMeasuresImageFilter()
dice_filter.Execute(gt, pred)

right_grac_dice_lower = dice_filter.GetDiceCoefficient()
print("Right gracilis lower dice:", right_grac_dice_lower)
hd_filter = sitk.HausdorffDistanceImageFilter()
hd_filter.Execute(gt, pred)

right_grac_hd_lower = hd_filter.GetHausdorffDistance()
print("Right gracilis lower Hausdorff distance:", right_grac_hd_lower)
results.append({
        #"image": gt_label,
        #"pred_label": pred_label,
        "Right gracilis lower dice:": right_grac_dice_lower,
        "Right gracilis lower Hausdorff distance:": right_grac_hd_lower
    })
df = pd.DataFrame(results)
print(df)